In [54]:
import json
import re
import xml.etree.ElementTree as ET

from pathlib import Path
from typing import Tuple

In [55]:
def get_filename(path: str | Path) -> str:
    return str(path).split("/")[-1]


def load_file(file_path: str | Path) -> str:
    with open(file_path, "r", encoding="latin1") as f:
        return f.read().encode("utf-8")


def clean_span(dirty_span: str) -> str:
    cleaner = "<.*?>"
    return re.sub(cleaner, "", str(dirty_span))


def parse_xml(raw_xml: ET.Element) -> str:
    text = ""
    for section in raw_xml:
        # Title is the span with ID '0'
        if "sid" in section.attrib and section.attrib["sid"] == "0":
            text += f"Title: {section.text}\n"
            continue

        # Extract the information from the rest of the paper
        section_tag = section.tag
        if section_tag == "ABSTRACT":
            text += "\nABSTRACT\n"
        else:
            attribs = section.attrib
            section_number = f" {attribs['number']}" if "number" in attribs else ""
            section_title = attribs["title"] if "title" in attribs else ""
            text += f"\n{section_tag}{section_number}: {section_title}\n"

        for span in section:
            text += f"{span.text}\n"
    return text


def load_xml(paper_idx: str, paper_path: Path) -> str:
    xml_path = paper_path.joinpath("Reference_XML", f"{paper_idx}.xml")
    raw_xml = ET.fromstring(load_file(xml_path))
    parsed_xml = parse_xml(raw_xml)
    return parsed_xml

In [56]:
def clean_abstract(raw_abstract: str) -> str:
    raw_spans = raw_abstract.splitlines()
    spans = [clean_span(rs.decode("utf-8")) for rs in raw_spans]
    return " ".join(spans)


def load_summaries(paper_path: Path) -> Tuple[str, str]:
    summ_path = paper_path.joinpath("summary")
    summ_files = [f for f in summ_path.iterdir()]
    for sf in summ_files:
        if "abstract" in get_filename(sf):
            raw_abstract = load_file(sf)
            abstract = clean_abstract(raw_abstract)
        if "human" in get_filename(sf):
            summary = load_file(sf).decode("utf-8")
    return abstract, summary

In [57]:
def get_papers_map(curr_dir: Path) -> dict[str:Path]:
    return {
        get_filename(paper_path): paper_path
        for paper_path in curr_dir.iterdir()
        if paper_path.is_dir()
    }

In [58]:
base_dir = Path("raw_data")
train_2019_path = base_dir.joinpath("Training-Set-2019")
others_path = [d for d in base_dir.iterdir() if d.is_dir() and d != train_2019_path]

In [59]:
# First parse all the data from 2019, Task 2
all_papers = {}
dirs_to_check = [d for d in train_2019_path.iterdir() if d.is_dir()]

for curr_dir in dirs_to_check:
    # Create map paper_id: Path
    papers_map = get_papers_map(curr_dir)

    # Extract data from each paper
    for paper_idx in papers_map:
        # Add new entry
        if paper_idx not in all_papers:
            all_papers[paper_idx] = {"full_text": "", "abstract": "", "human_summary": ""}
            all_papers[paper_idx]["full_text"] = load_xml(paper_idx, papers_map[paper_idx])
            abstract, summary = load_summaries(papers_map[paper_idx])
            all_papers[paper_idx]["abstract"] = abstract
            all_papers[paper_idx]["human_summary"] = summary

In [60]:
print(len(all_papers))

1018


In [61]:
# Then go and check in the past years if there are "new" documents
for other in others_path:
    papers_map = get_papers_map(other)

    # Extract data from each paper
    for paper_idx in papers_map:
        if paper_idx not in all_papers:
            all_papers[paper_idx] = {"full_text": "", "abstract": "", "human_summary": ""}
            all_papers[paper_idx]["full_text"] = load_xml(paper_idx, papers_map[paper_idx])
            abstract, summary = load_summaries(papers_map[paper_idx])
            all_papers[paper_idx]["abstract"] = abstract
            all_papers[paper_idx]["human_summary"] = summary

In [62]:
print(len(all_papers))

1038


In [63]:
# Save to disk
def append_to_jsonl(data: dict, file_path: str | Path) -> None:
    with open(file_path, "a") as fp:
        fp.write(json.dumps(data))
        fp.write("\n")


out_path = base_dir.joinpath("cl_scisumm.jsonl")
for paper_idx in all_papers:
    to_save = {"paper_id": paper_idx}
    to_save.update(all_papers[paper_idx])
    append_to_jsonl(to_save, out_path)